```text
Capital analysis
├── Import & Setup
├── PD and LGD estimation
└── Capital estimation
    ├── Capital comparison
    ├── Capital concentration
    └── Capital reduction
```

#### Import & Setup 

In [8]:
# Init work dir

from pathlib import Path
import sys
PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

In [9]:
# Standard modules

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns

# Framework

from src.feature_engineering import create_target_def12
from src.modelling import apply_pipe, transform_lgd
from src.preprocessing import my_input_load, ReplaceMinusOne
from src.validation import validate_predictions

# Configuration

pd.set_option("display.max_columns", None)
pd.set_option("display.width", None)
pd.set_option("display.expand_frame_repr", False)
from sklearn import set_config; set_config(transform_output="pandas")

In [10]:
# load input file

this_dataset = my_input_load(2021, 2021)
this_dataset, dr_summary = create_target_def12(this_dataset)

LoanDate range:     2021-01-01 00:00:00 2021-12-31 00:00:00
+12 months:         2022-01-01 00:00:00 2022-12-31 00:00:00
DefaultDate range:  2021-04-16 00:00:00 2023-10-13 00:00:00


LoanYear,num_loans,num_defaults,default_rate
2021,30266,3694,0.122051


## PD and LGD estimation

In [11]:
# import pipelines

import joblib
pipeline_pd  = joblib.load("../models/PIPELINE_ver_011_cal_shift-test.pkl")
pipeline_lgd = joblib.load("../models/lgd_model_Amount.pkl")

In [12]:
# apply PD pipeline

this_dataset_with_PD = apply_pipe(this_dataset, pipeline_pd)

Education: replacing 3 values of -1 with NaN
EmploymentStatus: replacing 30,266 values of -1 with NaN
MaritalStatus: replacing 30,266 values of -1 with NaN
OccupationArea: replacing 30,266 values of -1 with NaN
UseOfLoan: replacing 30,266 values of -1 with NaN


In [13]:
# apply LGD model

this_dataset_with_PD_LGD = transform_lgd(this_dataset_with_PD, pipeline_lgd)

In [14]:
# validate model output

validate_predictions(this_dataset_with_PD_LGD)

----------------------------------------
MODEL OUTPUT VALIDATION
----------------------------------------


,PD,LGD
Min,0.022,0.000
Mean,0.156,0.125
Max,0.640,0.224
NaNs,0.000,0.000


## Capital estimation

Next steps:

1. proxy: Capital = 2 * Expected Loss
1. Monte-Carlo method
1. proxy: Capital = 5 * Expected Loss